In [1]:
with open("text.txt", "r") as f:
    text = f.read()

- splitting words by (, ), emojies and etc....

In [2]:
import re 

test_text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', test_text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [3]:
import regex as re

def preprocess_fn(text):
    # First, replace newlines and tabs with spaces to normalize whitespace
    text = re.sub(r'[\n\t\r]+', ' ', text)
    
    # Collapse multiple spaces into one
    text = re.sub(r' +', ' ', text)
    
    # Now split on punctuation and special characters
    result = re.split(
        r'([- ,.:;?_!"()\'\\\+\(\)\*&\^%\$#@!<>?/]|\s|\p{Emoji})',
        text,
        flags=re.UNICODE
    )
    
    # Strip and filter empty tokens
    result = [item.strip() for item in result if item.strip()]
    
    return result

- Words to token Ids 

In [4]:
preprocess = preprocess_fn(text) # all words in original order
preprocess_len = len(preprocess)
all_words = sorted(set(preprocess)) # all words in sorted order 
vocab_size = len(all_words)

In [5]:
preprocess.extend(["<|endoftext|>", "<|unk|>"  ])
preprocess_len = len(preprocess)
all_tokens = sorted(list(set(preprocess)))
vocab_len = len(all_tokens)

In [6]:
vocab = {token:i for i, token in enumerate(all_tokens)}
vocab["<|endoftext|>"]

28

In [7]:
class Tokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        text = re.sub(r'[\n\t\r]+', ' ', text)
        text = re.sub(r' +', ' ', text)
        result = re.split(
            r'([- ,.:;?_!"()\'\\\+\(\)\*&\^%\$#@!<>?/]|\s|\p{Emoji})',
            text,
            flags=re.UNICODE
        )
        result = [item.strip() for item in result if item.strip()]
        result = [item if item in self.str_to_int else '<|unk|>' for item in result]

        
        ids = [self.str_to_int[i] for i in result]
        return ids 

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

In [8]:
tokenizer = Tokenizer(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace." 
text = " <|endoftext|> ".join((text1, text2))

ids = tokenizer.encode(text)
tokenizer.decode(ids)


'Hello, do you like tea? < <|unk|> > In the <|unk|> <|unk|> of the <|unk|>.'

In [9]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)


[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 13]


In [10]:
tokenizer.decode(integers)

'Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.'

In [11]:
s = 300
i = 50
ts = text[s:s+i]
ts

''

In [12]:
ints = tokenizer.encode(ts)
ints

[]

In [13]:
tokenizer.decode(ints)

''

## Data Sampling

In [14]:
with open("text.txt", "r") as f:
    raw_text = f.read()

small_text = raw_text[:15000] # too big for now

enc_text = tokenizer.encode(small_text)
print(len(enc_text))

5814


In [26]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_len, strde):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_len, "Number of tokenized inpit must at least be equal to max_len"

        for i in range(0, len(token_ids) -max_len, strde):
            input_chunks = token_ids[i:i+max_len]
            target_chunks = token_ids[i+1:i+max_len+1]

            self.input_ids.append(torch.tensor(input_chunks))
            self.target_ids.append(torch.tensor(target_chunks))


    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
        
    


In [27]:
def create_dataloader(txt, bs=4, max_len=256, stride=128, 
                      shuffle=True, drop_last=True, 
                      num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDataset(txt, tokenizer, max_len, stride)
    
    dataloader = DataLoader(
        dataset,
        batch_size=bs,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    ) 

    return dataloader

In [30]:
dataloader = create_dataloader(small_text, 1, 4, 1, False)
data_iter = iter(dataloader)
f_b = next(data_iter)
f_b

[tensor([[36479,  1095,   290,  3848]]), tensor([[1095,  290, 3848,  389]])]

### Token Embeddings


In [45]:
vocab_size = 50257 # total no. of tokens in gpt-2 tokenizer, byte-pair-encoding
out_dim = 256


token_embeddings_layer = torch.nn.Embedding(vocab_size, out_dim)

In [48]:
max_len = 4
data_loader = create_dataloader(small_text, bs=8, max_len=max_len, 
                                stride=max_len, shuffle=False)

data_iter = iter(data_loader)
inpts, trgts = next(data_iter)
inpts.shape, trgts.shape

(torch.Size([8, 4]), torch.Size([8, 4]))

In [51]:
token_embeddings = token_embeddings_layer(inpts)
token_embeddings.shape

torch.Size([8, 4, 256])

In [59]:
context_len = max_len
pos_embeddings_layer = torch.nn.Embedding(context_len, out_dim)

In [65]:
pos_embedding = pos_embeddings_layer.weight
pos_embedding.shape

torch.Size([4, 256])

In [68]:
input_embeddings = token_embeddings + pos_embedding
input_embeddings.shape

torch.Size([8, 4, 256])